<a href="https://colab.research.google.com/github/elliemci/agents/blob/main/monitoring_ealution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Observability and Evaluation of Agents

## Required Libraries

In [ ]:
%pip install 'smolagents[telemetry]'
%pip install opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents
%pip install langfuse datasets 'smolagents[gradio]'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/AgentsCourse

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/AgentsCourse


## Set Environmental variables and Connection to Langfuse OpenTelemetry

In [4]:
import os
import base64
from google.colab import userdata


langfuse_public_key = userdata.get('LANGFUSE_PUBLIC_KEY')
langfuse_secret_key = userdata.get('LANGFUSE_SECRET_KEY')

os.environ['LANGFUSE_PUBLIC_KEY'] = langfuse_public_key
os.environ['LANGFUSE_SECRET_KEY'] = langfuse_secret_key
os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com"

# Langfuse authentication - langfuse public and private keys are combined into
# the format public_key:secret_key and then encoded using encode() to prepare for
# base64 encoding which reoresents binary data in text format safe to transmit,
# the result of nase64 encoding is decoded back into a string
LANGFUSE_AUTH = base64.b64encode(
    f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()
).decode()

# set LangFuse OpenTelemetry endpoint URL environmental vareiable
# for sending OpenTelemetry logs, metrics, traces to Langfuse
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ.get("LANGFUSE_HOST") + "/api/public/otel"
# environment variable to configure HTTP headers for the OpenTelemetry data export;
# to ensure that the data is sent with proper authentication credentials the Autorization
# header is set to Basic and followed by the LANGFUSE_AUTh value calculated earlier
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

hugging_fase_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hugging_fase_token

## Setup a Trace-provider

In [5]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor


# create a TracerProvider for OpenTelemetry
trace_provider = TracerProvider()

# add a SimpleSpanProcessor with the OTLPSpanExporter to send traces
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

# set the global default tracer provider
trace.set_tracer_provider(trace_provider)
tracer = trace.get_tracer(__name__)

# Instrument smolagents with the configured provider
SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

## Test Instrumentation

Test the set up running a simple CodeAgent smolagents, should be able to see logs/spans in observability dashboard https://us.cloud.langfuse.com/ sign in with Google.

In [6]:
from smolagents import HfApiModel, CodeAgent

# instantiate a simple agent to test instrumentation
agent = CodeAgent(
    tools=[],
    model=HfApiModel()
)

request = "Write a short poem about the beauty of nature."

agent.run(request)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a short poem about the beauty of nature.                                                                  │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import random                                                                                                    
                                                                                                                   
  # Lists of words to be used in the poem                                                                          
  nouns = ["mountains", "rivers", "forests", "sunset", "clouds", "dawn", "ocean", "flowers", "trees"]              
  adjectives = ["majestic", "calming", "vast", "golden", "soft", "compelling", "azure", "bright", "whispering"]    
  verbs = ["stand", "flow", "sway", "paint", "conceal", "greet", "whisper", "shine", "breath"]                     
  objects = ["the valley", "the sky", "the earth", "the world", "the night", "the day", "the horizon", "heaven",   
  "the soul"]                                                                                                      
                                                                                                                   
  def generate_line():                                                                                             
      return f"The {random.choice(adjectives)} {random.choice(nouns)} {random.choice(verbs)} over                  
  {random.choice(objects)}."                                                                                       
                                                                                                                   
  poem = "\n".join(generate_line() for _ in range(4))                                                              
  print(poem)                                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The whispering clouds stand over the world.
The calming flowers sway over the soul.
The calming sunset stand over the horizon.
The vast flowers paint over heaven.

Out: None

[Step 1: Duration 14.67 seconds| Input tokens: 2,019 | Output tokens: 266]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import random                                                                                                    
                                                                                                                   
  # Lists of words to be used in the poem                                                                          
  nouns = ["mountains", "rivers", "forests", "sunset", "clouds", "dawn", "ocean", "flowers", "trees"]              
  adjectives = ["majestic", "calming", "vast", "golden", "soft", "compelling", "azure", "bright", "whispering"]    
  verbs = ["stand", "flow", "sway", "paint", "conceal", "greet", "whisper", "shine", "breathe"]                    
  objects = ["the valley", "the sky", "the earth", "the world", "the night", "the day", "the horizon", "heaven",   
  "the soul"]                                                                                                      
                                                                                                                   
  def generate_unique_line():                                                                                      
      adjective = random.choice(adjectives)                                                                        
      noun = random.choice(nouns)                                                                                  
      verb = random.choice(verbs)                                                                                  
      obj = random.choice(objects)                                                                                 
      # Ensure that the noun and object are not the same                                                           
      while noun in obj or obj in noun:                                                                            
          noun = random.choice(nouns)                                                                              
          obj = random.choice(objects)                                                                             
      return f"The {adjective} {noun} {verb} over {obj}."                                                          
                                                                                                                   
  poem = "\n".join(generate_unique_line() for _ in range(4))                                                       
  final_answer(poem)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The compelling clouds breathe over the night.
The majestic flowers breathe over the earth.
The compelling flowers shine over the night.
The compelling trees whisper over heaven.

[Step 2: Duration 20.23 seconds| Input tokens: 4,631 | Output tokens: 622]

'The compelling clouds breathe over the night.\nThe majestic flowers breathe over the earth.\nThe compelling flowers shine over the night.\nThe compelling trees whisper over heaven.'